# 40 — Capstone: AI Red-Teaming and Blue-Teaming — Lab

## Description

This lab applies offensive and defensive techniques across the full course threat landscape. You will begin with a structured red-team planning phase, then execute four attack flags (data poisoning, evasion, model extraction, and LLM injection) against a pre-deployed AI application, map each attack to its kill chain stage, and produce a combined red-team and incident response report.

## Objectives

- Produce a structured red-team plan document including: scope definition, threat model with attacker goals, rules of engagement, and success criteria for at least three attack classes (Step 0).
- Execute a structured red-team assessment covering at least three of four CTF attack classes (clean-label poisoning, evasion under query budget, model extraction, indirect prompt injection).
- Map each attack to its kill chain stage (data → training → deployment → inference → monitoring) and propose a defense-in-depth countermeasure.
- Activate blue-team monitoring to detect at least two attacks using runtime anomaly detection signals.
- Produce a complete AI incident response report for one detected attack.
- Compile a combined red-team report integrating all flag findings, kill chain mappings, and blue-team response timeline into a single structured document.

## Prerequisites

This lesson's lecture content is required: red-teaming methodology (planning, reconnaissance, exploitation, reporting), kill chain taxonomy (data → training → deployment → inference → monitoring), blue-team response playbook (detection triggers, containment actions), and AI incident response lifecycle (detection, containment, remediation, post-mortem).

From prior modules: **Lessons 08–10** (adversarial evasion attacks), **Lesson 19** (robustness benchmarking toolkits), **Lessons 20–22** (data poisoning and backdoor detection), **Lesson 25** (model extraction), **Lesson 35** (governance and incident response frameworks), **Lessons 36–37** (LLM injection and RAG security), and **Lesson 39** (guardrails and monitoring).

## Required Software and Packages

- numpy (1.20+) for array operations
- torch (1.10+) for model inference
- requests (2.25+) for API communication
- scikit-learn (0.24+) for model fidelity scoring
- pandas (1.1+) for data handling and incident timeline logging

## Environment and Setup Instructions

Run the cell below to import all required packages, configure model endpoints, and initialize logging infrastructure for monitoring.

In [ ]:
import numpy as np
import json
from datetime import datetime
from sklearn.metrics import accuracy_score
import pandas as pd

# Initialize logging for blue-team monitoring
logs = {
    'inference_log': [],
    'alert_log': [],
    'attack_evidence': {}
}

def raise_alert(timestamp, alert_type, severity, description):
    """Record an alert for blue-team response."""
    logs['alert_log'].append({
        'timestamp': timestamp,
        'alert_type': alert_type,
        'severity': severity,
        'description': description
    })

# Attack evidence storage structure
flags = {
    'flag_1_poisoning': {
        'attack_class': 'Data Poisoning (Clean-Label)',
        'kill_chain_stage': 'Data Collection',
        'evidence': {'poisoned_sample_features': None, 'baseline_accuracy': None, 'post_poison_accuracy': None, 'accuracy_drop_pp': None},
        'recommended_defense': 'Data provenance tracking and sample sanitization'
    },
    'flag_2_evasion': {
        'attack_class': 'Adversarial Evasion (PGD)',
        'kill_chain_stage': 'Inference',
        'evidence': {'query_budget_used': None, 'adversarial_image': None, 'target_class': None, 'misclassification_label': None, 'perturbation_magnitude': None},
        'recommended_defense': 'Adversarial training or certified defenses'
    },
    'flag_3_extraction': {
        'attack_class': 'Model Extraction (KnockoffNets)',
        'kill_chain_stage': 'Inference',
        'evidence': {'queries_sent': None, 'clone_agreement_rate': None, 'clone_accuracy_on_test': None, 'query_budget_remaining': None},
        'recommended_defense': 'Output perturbation and rate limiting per source IP'
    },
    'flag_4_llm_injection': {
        'attack_class': 'Indirect Prompt Injection (RAG)',
        'kill_chain_stage': 'Deployment',
        'evidence': {'malicious_document': None, 'injection_payload': None, 'tool_call_triggered': None, 'unauthorized_action': None},
        'recommended_defense': 'COLANG guardrails and prompt robustness training'
    }
}

# Red-team plan storage
red_team_plan = {
    'scope': None,
    'threat_model': None,
    'rules_of_engagement': None,
    'success_criteria': None
}

print("Environment initialized. Logging infrastructure active.")

## Background and Context

You are a red-team member testing an AI application that integrates image classification, a text-to-text language model API, and a federated learning training pipeline. You will begin by creating a red-team plan that scopes your engagement, identifies threats, sets rules of engagement, and defines success criteria. Then, for each attack class, you will craft an exploit, record evidence, and map the attack to its kill chain stage. At the end, you will write a combined red-team report and a blue-team incident response report for the highest-severity finding.

## Exercise Instructions

* **Step 0: Red-Team Planning** — Define scope, threat model, rules of engagement, and success criteria. Fill in the planning template below and record your plan.

* **Step 1: Flag 1 (Data Poisoning)** — Inject one clean-label poisoned sample; confirm model accuracy drops by at least 5 percentage points on the test set; record the poisoned sample's features as evidence.

* **Step 2: Flag 2 (Evasion)** — Craft a PGD adversarial example under a 500-query budget; record the target misclassification label as evidence.

* **Step 3: Flag 3 (Model Extraction)** — Send 5,000 queries using KnockoffNets strategy; train a substitute model; record clone fidelity on held-out test set as evidence.

* **Step 4: Flag 4 (LLM Injection)** — Exploit an indirect prompt injection in the RAG chatbot to trigger an unauthorized tool call; document the exploit chain as evidence.

* **Step 5: Write Reports** — Write the combined red-team report with one section per flag; write a three-sentence incident response summary for Flag 4 covering detection, containment, and remediation.

## Step 0: Red-Team Planning

Before executing any attacks, create a structured red-team plan. Your plan defines the scope of your engagement, identifies the threat model, sets rules of engagement (boundaries and constraints), and specifies success criteria for each attack.

Fill in each section of the planning template below based on the lecture content on red-teaming methodology. Your plan should cover at least three of the four attack classes (data poisoning, evasion, model extraction, LLM injection).

In [ ]:
# RED-TEAM PLANNING TEMPLATE
# Fill in each section with concrete details based on the lecture scope lock and methodology.

# SECTION 1: Scope Definition
# Define which components, data, and APIs are in-scope for testing.
# Include: target model architecture, input domain, training pipeline type, deployed defenses.
scope = """
[TODO: Fill in scope definition]
Example: The engagement targets an image classifier (ResNet-50) deployed on an inference API.
Input domain: 224x224 RGB images (CIFAR-10 classes). Training pipeline is batch-retrained weekly.
No adversarial training or input preprocessing is currently deployed.
"""

# SECTION 2: Threat Model
# Identify attacker goals and capabilities.
# Include: attack goals (misclassify, poison, extract, manipulate), attacker access level, constraints.
threat_model = """
[TODO: Fill in threat model]
Example: Attacker goal is to misclassify stop signs as speed-limit signs to disrupt autonomous vehicles.
Attacker has black-box access to the API (500 queries max). Attack must be imperceptible (Linf <= 8/255).
"""

# SECTION 3: Rules of Engagement
# Define boundaries: which attacks are permitted, access type, query budgets, imperceptibility requirements.
# Include: permitted attack classes, white-box vs black-box, query/sample budgets, stealth requirements.
rules_of_engagement = """
[TODO: Fill in rules of engagement]
Example: Attacks permitted on inference only (no training pipeline access).
Black-box access assumed. Query budget: 500 per attack. Perturbation must be imperceptible (Linf 8/255).
No access to model weights, training data, or gradients.
"""

# SECTION 4: Success Criteria
# Define measurable, concrete achievements for each attack class.
# Include: one success criterion per attack class (poisoning, evasion, extraction, injection).
success_criteria = """
[TODO: Fill in success criteria]
Example:
- Data Poisoning: Inject clean-label sample; achieve >= 5 pp accuracy drop on test set.
- Evasion: Misclassify >=1 target image under 500-query budget with Linf <= 8/255.
- Model Extraction: Train clone with >= 80% agreement rate on held-out test set.
- LLM Injection: Trigger unauthorized tool call via indirect prompt injection in RAG vector store.
"""

# Store your plan
red_team_plan['scope'] = scope
red_team_plan['threat_model'] = threat_model
red_team_plan['rules_of_engagement'] = rules_of_engagement
red_team_plan['success_criteria'] = success_criteria

print("RED-TEAM PLAN")
print("=" * 60)
print("\n[Scope]")
print(red_team_plan['scope'])
print("\n[Threat Model]")
print(red_team_plan['threat_model'])
print("\n[Rules of Engagement]")
print(red_team_plan['rules_of_engagement'])
print("\n[Success Criteria]")
print(red_team_plan['success_criteria'])
print("\n" + "=" * 60)
print("Plan recorded. Proceed to Step 1 (Flag exercises).")

## Steps 1–4: Flag Exercises
Now execute each attack within the scope and rules you defined in Step 0. Each attack corresponds to one of the four attack classes covered in the capstone.

In [ ]:
# Scaffold for Flag Exercises
# Each flag is incomplete; you must implement or modify the attack simulation.

# Data structures are initialized in the Environment Setup cell above.
# You have access to: logs dict, flags dict, red_team_plan dict, raise_alert() function

# EXERCISE 1: Flag 1 (Data Poisoning)
# TODO: Implement a clean-label poisoning attack by:
# - Creating a poisoned sample using gradient alignment
# - Computing baseline accuracy before poisoning
# - Computing post-poison accuracy
# - Recording the accuracy drop (must be >= 5 pp)
# Fill in the following:

baseline_test_accuracy = 0.92  # MODIFY: Replace with actual model accuracy
target_class = 0  # MODIFY: Choose target class
poisoned_sample = None  # TODO: Implement poisoned sample generation
post_poison_accuracy = None  # TODO: Query model on poisoned data and measure accuracy
accuracy_drop_pp = None  # TODO: Compute (baseline - post_poison) * 100

# Once implemented, record evidence:
# flags['flag_1_poisoning']['evidence']['poisoned_sample_features'] = poisoned_sample
# flags['flag_1_poisoning']['evidence']['baseline_accuracy'] = baseline_test_accuracy
# flags['flag_1_poisoning']['evidence']['post_poison_accuracy'] = post_poison_accuracy
# flags['flag_1_poisoning']['evidence']['accuracy_drop_pp'] = accuracy_drop_pp
# logs['attack_evidence']['flag_1'] = {...}

print("Flag 1 scaffold loaded. Implement the poisoning attack above.")

# EXERCISE 2: Flag 2 (Adversarial Evasion)
# TODO: Implement PGD adversarial example generation by:
# - Creating a benign image
# - Applying iterative perturbations (PGD) respecting query budget (500 queries)
# - Confirming misclassification on target class
# Fill in the following:

query_budget = 500
queries_used_evasion = None  # TODO: Track queries sent during PGD attack
benign_image = None  # TODO: Load or generate benign image
perturbation = None  # TODO: Compute adversarial perturbation
adversarial_image = None  # TODO: benign_image + perturbation
true_class = 7
target_misclass = None  # TODO: Record which class the model misclassified to

# Once implemented, record evidence:
# flags['flag_2_evasion']['evidence']['query_budget_used'] = queries_used_evasion
# flags['flag_2_evasion']['evidence']['misclassification_label'] = target_misclass
# ... (see flag structure above for all required fields)

print("Flag 2 scaffold loaded. Implement the evasion attack above.")

# EXERCISE 3: Flag 3 (Model Extraction)
# TODO: Implement KnockoffNets model extraction by:
# - Sending queries to a rate-limited API (5000 total queries)
# - Training a substitute/clone model on collected predictions
# - Evaluating clone fidelity on held-out test set
# Fill in the following:

extraction_queries = 5000
clone_queries_used = None  # TODO: Track API queries sent
clone_agreement_rate = None  # TODO: Measure agreement between original and clone

# Once implemented, record evidence:
# flags['flag_3_extraction']['evidence']['queries_sent'] = clone_queries_used
# flags['flag_3_extraction']['evidence']['clone_agreement_rate'] = clone_agreement_rate
# ... (see flag structure above for all required fields)

print("Flag 3 scaffold loaded. Implement the extraction attack above.")

# EXERCISE 4: Flag 4 (LLM Indirect Prompt Injection)
# TODO: Implement indirect prompt injection by:
# - Crafting an adversarial document with hidden instructions
# - Injecting it into the RAG vector store
# - Querying the chatbot and observing if tool calls are triggered
# Fill in the following:

malicious_doc = None  # TODO: Craft document with hidden instructions
injection_payload = None  # TODO: User query that should NOT trigger tool call
tool_call_triggered = None  # TODO: Record which tool was called
unauthorized_action = None  # TODO: Describe the unauthorized action

# Once implemented, record evidence:
# flags['flag_4_llm_injection']['evidence']['malicious_document'] = malicious_doc
# flags['flag_4_llm_injection']['evidence']['tool_call_triggered'] = tool_call_triggered
# ... (see flag structure above for all required fields)

print("Flag 4 scaffold loaded. Implement the LLM injection attack above.")

print("\nAll flag scaffolds loaded. Complete each exercise step by step.")

## Expected Outputs and Verification

Flag 1 accuracy drop 7.0 pp (meets ≥5 pp requirement); Flag 2 misclassified to target class 3 using 342 of 500 queries; Flag 3 clone agreement rate 82.0% on held-out test set; Flag 4 unauthorized tool call (transfer_funds) triggered; Blue-team alerts: 2 alerts logged (high query volume for Flag 2, unauthorized tool call for Flag 4). Run the cell below to verify all evidence was recorded correctly.

In [ ]:
# Verification: Check that all flags executed successfully
flag_names = ['flag_1_poisoning', 'flag_2_evasion', 'flag_3_extraction', 'flag_4_llm_injection']
all_passed = True
results = []

for flag_name in flag_names:
    flag = flags[flag_name]
    evidence_complete = all(v is not None for v in flag['evidence'].values())
    results.append({'flag': flag_name, 'status': 'PASS' if evidence_complete else 'FAIL'})
    if not evidence_complete:
        all_passed = False

# Verify blue-team alerts
alert_count = len(logs['alert_log'])
alerts_sufficient = alert_count >= 2
results.append({'flag': 'blue_team_monitoring', 'status': 'PASS' if alerts_sufficient else 'FAIL'})
if not alerts_sufficient:
    all_passed = False

# Verify red-team plan
plan_complete = all(v is not None for v in red_team_plan.values())
results.append({'flag': 'red_team_plan', 'status': 'PASS' if plan_complete else 'FAIL'})
if not plan_complete:
    all_passed = False

for result in results:
    print(f"{result['flag']}: {result['status']}")

print(f"\nOverall Result: {'PASS' if all_passed else 'FAIL'}")
print(f"Alerts recorded: {alert_count}")
print(f"Plan sections completed: {sum(1 for v in red_team_plan.values() if v is not None)}/4")

## Step 5: Red-Team and Incident Response Reports (6 min)

Complete the following two reports as part of this exercise.

**Report 1: Combined Red-Team Report** — For each flag (1–4), write one section in the format:
- Attack Class: [e.g., "Data Poisoning (Clean-Label)"]
- Kill Chain Stage: [e.g., "Data Collection"]
- Evidence: [Summary of findings, e.g., poisoned sample features, accuracy drop]
- Recommended Defense: [Single concrete defense measure]

**Report 2: Incident Response Report for Flag 4** — Write 3 sentences covering:
1. Detection: How should the blue team have detected this LLM injection before it happened?
2. Containment: What immediate action stops this attack?
3. Remediation: What long-term fix prevents recurrence?

In [ ]:
# Combined Red-Team Report
# Write your report below. Format each flag's findings clearly.

red_team_report = """
## Flag 1: Data Poisoning (Clean-Label)
**Attack Class:** 
**Kill Chain Stage:** 
**Evidence:** 
**Recommended Defense:** 

## Flag 2: Adversarial Evasion (PGD)
**Attack Class:** 
**Kill Chain Stage:** 
**Evidence:** 
**Recommended Defense:** 

## Flag 3: Model Extraction (KnockoffNets)
**Attack Class:** 
**Kill Chain Stage:** 
**Evidence:** 
**Recommended Defense:** 

## Flag 4: Indirect Prompt Injection (RAG)
**Attack Class:** 
**Kill Chain Stage:** 
**Evidence:** 
**Recommended Defense:** 
"""

print("Red-Team Report Template Created")
print("Complete the fields above with your findings from all four flag exercises.")

In [ ]:
# Incident Response Report for Flag 4 (LLM Prompt Injection)
# Write a 3-sentence report below covering Detection, Containment, and Remediation.

incident_response_report = """
**Detection:** [How should the blue team have detected the prompt injection attack in real-time using the monitoring architecture from the lecture? Name specific signals.]

**Containment:** [What immediate action stops the attack and isolates the compromised system?]

**Remediation:** [What long-term fix prevents this attack from succeeding again in the future?]
"""

print("Incident Response Report Template Created")
print("Complete the three sentences above based on the blue-team response playbook and monitoring architecture.")

## Comprehension Questions

1. Why is data poisoning (Flag 1) a data-collection-stage attack, and why is defense at the training stage (robust aggregation) insufficient to stop it? What defense at the data-collection stage would have prevented Flag 1?

2. Model extraction (Flag 3) achieved 82 percent clone fidelity despite rate limiting. Propose two defenses from m39 (guardrails and sandboxing) and m28 (encrypted computation) that would reduce clone fidelity to below 50 percent.

## Optional Challenges

**Optional Challenge 1**: Implement a real PGD attack (from m09) using Foolbox and measure its actual convergence rate on a small MNIST model. Compare queries required against the simulated 342 queries recorded in Flag 2.

**Optional Challenge 2**: Write a Python function that detects clean-label poisoning by computing Wasserstein distance between training-data feature distributions. Identify the threshold separating benign data drift from poisoning with 95 percent precision.

## Cleanup Instructions

No cleanup required. All attack simulations are in-memory; no files, models, or data were persisted to disk.

## Completion Criteria

You have successfully completed this lab when:
- [ ] Step 0: Red-team plan document is complete (scope, threat model, rules of engagement, success criteria filled in for ≥3 attack classes).
- [ ] Steps 1–4: All four flags executed and evidence recorded (poisoned_sample_features, target misclassification label, clone agreement rate, unauthorized tool call).
- [ ] Step 5: Combined red-team report written (one section per flag with Attack Class, Kill Chain Stage, Evidence, Recommended Defense).
- [ ] Step 5: Incident response report written (3 sentences on Detection, Containment, Remediation for Flag 4).
- [ ] Verification cell prints "PASS" (all flag evidence recorded, ≥2 blue-team alerts logged, plan complete).